<a href="https://colab.research.google.com/github/DiaaEddin963/ACE6313-Employee-Attrition-Group-I/blob/main/ACE6313_Group_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
# --- PART A: DATA PREPROCESSING By Diaa ---
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Starting Part A: Data Preprocessing Pipeline...\n" + "-"*40)

# ==========================================
# 1. DATA CLEANING
# ==========================================
url = 'https://raw.githubusercontent.com/DiaaEddin963/ACE6313-Employee-Attrition-Group-I/main/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(url)

# A. Handle Missing Values & Duplicates
df = df.dropna().drop_duplicates()

# B. Handle Inconsistencies (Drop uninformative features)
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_cleaned = df.drop(columns=cols_to_drop, errors='ignore')

# C. Handle Outliers (Using IQR method to cap extreme Monthly Incomes)
Q1 = df_cleaned['MonthlyIncome'].quantile(0.25)
Q3 = df_cleaned['MonthlyIncome'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
# Cap outliers instead of dropping them to preserve valuable minority class data
df_cleaned.loc[df_cleaned['MonthlyIncome'] > upper_bound, 'MonthlyIncome'] = upper_bound

print(f"1. Data Cleaning Complete.")
print(f"   -> Handled missing values, duplicates, inconsistencies, and capped Outliers.")
print(f"   -> Current Shape: {df_cleaned.shape}")

# ==========================================
# 2. DATA TRANSFORMATION
# ==========================================
# A. Feature Engineering (Creating a meaningful new metric)
# Creating a 'Tenure-to-Age Ratio' to see if employees who spent their whole youth here leave less
df_cleaned['Tenure_Age_Ratio'] = df_cleaned['YearsAtCompany'] / df_cleaned['Age']

# B. Target Separation
y = df_cleaned['Attrition'].map({'Yes': 1, 'No': 0})
X = df_cleaned.drop('Attrition', axis=1)

# C. Feature Encoding (Categorical to Binary)
X_encoded = pd.get_dummies(X, drop_first=True)

# D. Feature Scaling (Standardizing numerical ranges)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_transformed = pd.DataFrame(X_scaled, columns=X_encoded.columns)

print(f"\n2. Data Transformation Complete.")
print(f"   -> Performed Feature Engineering, One-Hot Encoding, and Standard Scaling.")
print(f"   -> Current Shape: {X_transformed.shape}")

# ==========================================
# 3. DATA REDUCTION (Feature Selection)
# ==========================================
from sklearn.feature_selection import SelectKBest, f_classif

# Select the top 20 most predictive features instead of using PCA
# This preserves feature meaning and tree-based model performance
selector = SelectKBest(score_func=f_classif, k=20)
X_reduced = selector.fit_transform(X_scaled, y)

# Keep the data as a DataFrame to retain column names for HR interpretation
selected_columns = X_transformed.columns[selector.get_support()]
X_final = pd.DataFrame(X_reduced, columns=selected_columns)

print(f"\n3. Data Reduction Complete.")
print(f"   -> Applied Feature Selection (SelectKBest).")
print(f"   -> Reduced features from {X_transformed.shape[1]} down to {X_final.shape[1]} most critical features.")
print("-" * 40 + "\nPART A FULLY COMPLETED. DATA IS READY FOR PART B.")

Starting Part A: Data Preprocessing Pipeline...
----------------------------------------
1. Data Cleaning Complete.
   -> Handled missing values, duplicates, inconsistencies, and capped Outliers.
   -> Current Shape: (1470, 31)

2. Data Transformation Complete.
   -> Performed Feature Engineering, One-Hot Encoding, and Standard Scaling.
   -> Current Shape: (1470, 45)

3. Data Reduction Complete.
   -> Applied Feature Selection (SelectKBest).
   -> Reduced features from 45 down to 20 most critical features.
----------------------------------------
PART A FULLY COMPLETED. DATA IS READY FOR PART B.


In [35]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
import pandas as pd

print("--- PREPARING DATA FOR MODELING ---")
# We use stratify=y to ensure the train and test sets have the same ratio of Yes/No attrition.
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

--- PREPARING DATA FOR MODELING ---
Training set size: 1176
Testing set size: 294


In [36]:
from sklearn.linear_model import LogisticRegression

print("--- 1. LOGISTIC REGRESSION ---")
lr = LogisticRegression(class_weight='balanced',max_iter=1000, random_state=42)

# Define hyperparameters to tune
lr_params = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs", "liblinear"]
}

# Setup GridSearchCV optimizing for F1-score
lr_grid = GridSearchCV(lr, param_grid=lr_params, cv=5, scoring='f1', n_jobs=-1)
lr_grid.fit(X_train, y_train)

# Evaluate the best model
print(f"Best Hyperparameters: {lr_grid.best_params_}")
y_pred_lr = lr_grid.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))


--- 1. LOGISTIC REGRESSION ---
Best Hyperparameters: {'C': 0.1, 'solver': 'liblinear'}

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.79      0.86       247
           1       0.39      0.70      0.50        47

    accuracy                           0.78       294
   macro avg       0.66      0.75      0.68       294
weighted avg       0.85      0.78      0.80       294



In [37]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

print("--- 5. K-NEAREST NEIGHBORS (kNN) ---")
knn = KNeighborsClassifier()

# Define hyperparameters to tune
knn_params = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

# Setup GridSearchCV optimizing for F1-score
knn_grid = GridSearchCV(knn, param_grid=knn_params, cv=5, scoring='f1', n_jobs=-1)
knn_grid.fit(X_train, y_train)
best_knn = knn_grid.best_estimator_

print(f"Best Hyperparameters: {knn_grid.best_params_}")

# Standard Evaluation (Threshold = 0.5)
print("\n--- Standard Classification Report (Threshold 0.5) ---")
y_pred_knn_standard = best_knn.predict(X_test)
print(classification_report(y_test, y_pred_knn_standard))

# --- THE FIX: Custom Thresholding ---
print("\n--- Adjusted Classification Report (Threshold 0.2) ---")
# Instead of 1 or 0, predict_proba gets the % chance of being in class 1
y_proba_knn = best_knn.predict_proba(X_test)[:, 1]

# If the probability is >= 20%, we classify them as "Yes" (1)
custom_threshold = 0.2
y_pred_knn_adjusted = (y_proba_knn >= custom_threshold).astype(int)

print(classification_report(y_test, y_pred_knn_adjusted))


--- 5. K-NEAREST NEIGHBORS (kNN) ---
Best Hyperparameters: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'uniform'}

--- Standard Classification Report (Threshold 0.5) ---
              precision    recall  f1-score   support

           0       0.85      0.96      0.90       247
           1       0.36      0.11      0.16        47

    accuracy                           0.83       294
   macro avg       0.60      0.53      0.53       294
weighted avg       0.77      0.83      0.79       294


--- Adjusted Classification Report (Threshold 0.2) ---
              precision    recall  f1-score   support

           0       0.86      0.73      0.79       247
           1       0.21      0.38      0.27        47

    accuracy                           0.67       294
   macro avg       0.54      0.56      0.53       294
weighted avg       0.76      0.67      0.71       294



In [38]:
from sklearn.svm import SVC

print("--- 4. SUPPORT VECTOR MACHINE (SVM) - EXPANDED TUNING ---")
# We remove class_weight here because we will tune it in the grid!
svm = SVC(random_state=42, probability=True)

# Expanded hyperparameter grid
svm_params = {
    "C": [0.1, 1, 10, 50, 100],
    "kernel": ["rbf"], # We dropped 'linear' since it already chose 'rbf' earlier
    "gamma": ["scale", "auto", 0.01, 0.1],
    # Testing 'balanced' against manual 2x, 3x, and 4x penalties for Class 1
    "class_weight": ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}]
}

# Setup GridSearchCV optimizing for F1-score
svm_grid = GridSearchCV(svm, param_grid=svm_params, cv=5, scoring='f1', n_jobs=-1)
svm_grid.fit(X_train, y_train)

# Evaluate the best model
print(f"Best Hyperparameters: {svm_grid.best_params_}")
y_pred_svm = svm_grid.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))


--- 4. SUPPORT VECTOR MACHINE (SVM) - EXPANDED TUNING ---
Best Hyperparameters: {'C': 1, 'class_weight': {0: 1, 1: 4}, 'gamma': 0.01, 'kernel': 'rbf'}

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.86      0.89       247
           1       0.44      0.57      0.50        47

    accuracy                           0.81       294
   macro avg       0.67      0.72      0.69       294
weighted avg       0.84      0.81      0.82       294

